In [15]:
# Install the required spaCy model (run this cell first)
!python -m spacy download en_core_web_lg

import pandas as pd
import numpy as np
import re
import spacy
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load and clean the data
df = pd.read_csv('/content/train.csv', engine='python', on_bad_lines='skip')
df = df.dropna()
df = df.drop(['id', 'qid1', 'qid2'], axis=1)

# Sample 80,000 rows to manage memory
new_df = df.sample(50000, random_state=2)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [16]:
def preprocess(q):
    q = str(q).lower().strip()

    # Replace certain special characters with their string equivalents
    q = q.replace('%', ' percent')
    q = q.replace('$', ' dollar ')
    q = q.replace('₹', ' rupee ')
    q = q.replace('€', ' euro ')
    q = q.replace('@', ' at ')

    # Remove math tags
    q = q.replace('[math]', '')

    # Replace large numbers
    q = q.replace(',000,000,000 ', 'b ')
    q = q.replace(',000,000 ', 'm ')
    q = q.replace(',000 ', 'k ')

    # Decontracting words
    contractions = {
        "ain't": "am not", "aren't": "are not", "can't": "can not", "can't've": "can not have",
        "'cause": "because", "could've": "could have", "couldn't": "could not", "couldn't've": "could not have",
        "didn't": "did not", "doesn't": "does not", "don't": "do not", "hadn't": "had not",
        "hadn't've": "had not have", "hasn't": "has not", "haven't": "have not", "he'd": "he would",
        "he'd've": "he would have", "he'll": "he will", "he'll've": "he will have", "he's": "he is",
        "how'd": "how did", "how'd'y": "how do you", "how'll": "how will", "how's": "how is",
        "i'd": "i would", "i'd've": "i would have", "i'll": "i will", "i'll've": "i will have",
        "i'm": "i am", "i've": "i have", "isn't": "is not", "it'd": "it would", "it'd've": "it would have",
        "it'll": "it will", "it'll've": "it will have", "it's": "it is", "let's": "let us", "ma'am": "madam",
        "mayn't": "may not", "might've": "might have", "mightn't": "might not", "mightn't've": "might not have",
        "must've": "must have", "mustn't": "must not", "mustn't've": "must not have", "needn't": "need not",
        "needn't've": "need not have", "o'clock": "of the clock", "oughtn't": "ought not",
        "oughtn't've": "ought not have", "shan't": "shall not", "sha'n't": "shall not",
        "shan't've": "shall not have", "she'd": "she would", "she'd've": "she would have",
        "she'll": "she will", "she'll've": "she will have", "she's": "she is", "should've": "should have",
        "shouldn't": "should not", "shouldn't've": "should not have", "so've": "so have", "so's": "so as",
        "that'd": "that would", "that'd've": "that would have", "that's": "that is", "there'd": "there would",
        "there'd've": "there would have", "there's": "there is", "they'd": "they would",
        "they'd've": "they would have", "they'll": "they will", "they'll've": "they will have",
        "they're": "they are", "they've": "they have", "to've": "to have", "wasn't": "was not",
        "we'd": "we would", "we'd've": "we would have", "we'll": "we will", "we'll've": "we will have",
        "we're": "we are", "we've": "we have", "weren't": "were not", "what'll": "what will",
        "what'll've": "what will have", "what're": "what are", "what's": "what is", "what've": "what have",
        "when's": "when is", "when've": "when have", "where'd": "where did", "where's": "where is",
        "where've": "where have", "who'll": "who will", "who'll've": "who will have", "who's": "who is",
        "who've": "who have", "why's": "why is", "why've": "why have", "will've": "will have",
        "won't": "will not", "won't've": "will not have", "would've": "would have", "wouldn't": "would not",
        "wouldn't've": "would not have", "y'all": "you all", "y'all'd": "you all would",
        "y'all'd've": "you all would have", "y'all're": "you all are", "y'all've": "you all have",
        "you'd": "you would", "you'd've": "you would have", "you'll": "you will", "you'll've": "you will have",
        "you're": "you are", "you've": "you have"
    }

    q_decontracted = []
    for word in q.split():
        if word in contractions:
            word = contractions[word]
        q_decontracted.append(word)

    q = ' '.join(q_decontracted)
    q = q.replace("'ve", " have")
    q = q.replace("n't", " not")
    q = q.replace("'re", " are")
    q = q.replace("'ll", " will")

    # Remove HTML tags safely
    q = BeautifulSoup(q, "html.parser").get_text()

    # Remove punctuation (Keep only alphanumeric characters and spaces)
    q = re.sub(r'[^\w\s]', ' ', q)

    # Remove extra whitespace
    q = re.sub(r'\s+', ' ', q).strip()

    return q

print("Preprocessing data...")
new_df['question1'] = new_df['question1'].apply(preprocess)
new_df['question2'] = new_df['question2'].apply(preprocess)

Preprocessing data...


In [17]:
X=df.drop('is_duplicate',axis=1)
y=df['is_duplicate']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=1)

In [18]:
!pip install fuzzywuzzy python-Levenshtein scipy

from fuzzywuzzy import fuzz
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm
import numpy as np
import scipy.spatial.distance as dist

print("Loading spaCy model...")
nlp = spacy.load("en_core_web_lg")

# 1. Fit TF-IDF on training data
print("Fitting TF-IDF on training data...")
questions = list(X_train['question1']) + list(X_train['question2'])
tfidf = TfidfVectorizer(lowercase=False)
tfidf.fit(questions)
idf_dict = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

def get_weighted_sentence_vector(text):
    doc = nlp(text)
    vec = np.zeros(300)
    weight_sum = 0
    for token in doc:
        word = token.text
        if word in idf_dict and token.has_vector:
            weight = idf_dict[word]
            vec += token.vector * weight
            weight_sum += weight

    if weight_sum != 0:
        vec /= weight_sum
    return vec

# 2. Extract both Semantic Distances and Fuzzy Features
def extract_advanced_features(df):
    features = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting Features"):
        q1 = str(row['question1'])
        q2 = str(row['question2'])

        # Fuzzy Lexical Features
        fuzzy_feats = [
            fuzz.ratio(q1, q2),
            fuzz.partial_ratio(q1, q2),
            fuzz.token_sort_ratio(q1, q2),
            fuzz.token_set_ratio(q1, q2)
        ]

        # Semantic Vector Features
        vec1 = get_weighted_sentence_vector(q1)
        vec2 = get_weighted_sentence_vector(q2)

        if np.all(vec1 == 0) or np.all(vec2 == 0):
            semantic_feats = [0.0, 0.0, 0.0]
        else:
            semantic_feats = [
                1 - dist.cosine(vec1, vec2), # Cosine Similarity (1 is identical)
                dist.euclidean(vec1, vec2),  # Euclidean Distance
                dist.cityblock(vec1, vec2)   # Manhattan Distance
            ]

        features.append(fuzzy_feats + semantic_feats)

    return np.array(features)

print("Vectorizing train data...")
# Your new feature matrix is now 7 highly-informative dimensions instead of 604
X_train_vec = extract_advanced_features(X_train)

print("Vectorizing test data...")
X_test_vec = extract_advanced_features(X_test)

print(f"Final training data shape: {X_train_vec.shape}")

Loading spaCy model...
Fitting TF-IDF on training data...
Vectorizing train data...


Extracting Features: 100%|██████████| 323429/323429 [1:30:54<00:00, 59.30it/s]


Vectorizing test data...


Extracting Features: 100%|██████████| 80858/80858 [23:12<00:00, 58.08it/s]


Final training data shape: (323429, 7)


In [19]:
# Define models
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        eval_metric='logloss'
    )
}

results = []

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_vec, y_train)

    y_pred = model.predict(X_test_vec)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='binary')
    recall = recall_score(y_test, y_pred, average='binary')
    f1 = f1_score(y_test, y_pred, average='binary')

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

results_df = pd.DataFrame(results)
print("\nEvaluation Results:")
print(results_df)



Training Random Forest...
Training XGBoost...

Evaluation Results:
           Model  Accuracy  Precision    Recall  F1 Score
0  Random Forest  0.736278   0.639856  0.651744  0.645745
1        XGBoost  0.739556   0.636545  0.684809  0.659795


In [20]:
# Set the final model to Random Forest for inference
final_model = models["XGBoost"]

In [21]:
def predict_duplicate_w2v_tf_fuzzy(q1, q2, model):
    # Preprocess questions
    clean_q1 = preprocess(q1)
    clean_q2 = preprocess(q2)

    # Get TF-IDF weighted 300D vectors
    vec1 = get_weighted_sentence_vector(clean_q1)
    vec2 = get_weighted_sentence_vector(clean_q2)

    # Calculate Semantic Distances
    if np.all(vec1 == 0) or np.all(vec2 == 0):
        semantic_feats = [0.0, 0.0, 0.0]
    else:
        semantic_feats = [
            1 - dist.cosine(vec1, vec2),
            dist.euclidean(vec1, vec2),
            dist.cityblock(vec1, vec2)
        ]

    # Calculate Fuzzy Features
    fuzzy_feats = [
        fuzz.ratio(clean_q1, clean_q2),
        fuzz.partial_ratio(clean_q1, clean_q2),
        fuzz.token_sort_ratio(clean_q1, clean_q2),
        fuzz.token_set_ratio(clean_q1, clean_q2)
    ]

    # Combine into 7D array
    query_vec = np.array([fuzzy_feats + semantic_feats])

    # Predict
    prediction = model.predict(query_vec)[0]
    probability = model.predict_proba(query_vec)[0]

    if prediction == 1:
        return f"Duplicate (Confidence: {probability[1]:.2%})"
    else:
        return f"Not Duplicate (Confidence: {probability[0]:.2%})"

In [22]:
question_pairs = [
    ("How can I learn Python?", "What is the best way to learn Python?"),
    ("What is machine learning?", "What is deep learning?"),
    ("What is the capital of India?", "What is the population of India?"),
    ("How do I learn data science?", "How do I get a job in data science?"),
    (
        "How can I learn Python?",
        "What is the best way to learn Python?"
    ),
    (
        "How can I lose weight?",
        "What are the best ways to lose weight?"
    ),
    (
        "What is machine learning?",
        "Can someone explain machine learning?"
    ),
    (
        "How do I learn data science?",
        "What is the best way to learn data science?"
    ),
    (
        "How can I improve my English?",
        "What are the best ways to improve my English?"
    ),

    # Not duplicate / Different meaning
    (
        "How can I learn Python?",
        "How can I learn Java?"
    ),
    (
        "What is machine learning?",
        "What is deep learning?"
    ),
    (
        "How can I lose weight?",
        "How can I gain weight?"
    ),
    (
        "What is the capital of India?",
        "What is the population of India?"
    ),
    (
        "How do I learn data science?",
        "How do I get a job in data science?"
    )
]

In [23]:
i=1
for q1,q2 in question_pairs:
  print(i,predict_duplicate_w2v_tf_fuzzy(q1, q2, final_model))
  i+=1

1 Duplicate (Confidence: 60.97%)
2 Not Duplicate (Confidence: 70.00%)
3 Not Duplicate (Confidence: 55.14%)
4 Duplicate (Confidence: 68.06%)
5 Duplicate (Confidence: 60.97%)
6 Not Duplicate (Confidence: 51.40%)
7 Duplicate (Confidence: 51.81%)
8 Duplicate (Confidence: 60.57%)
9 Duplicate (Confidence: 71.84%)
10 Not Duplicate (Confidence: 76.11%)
11 Not Duplicate (Confidence: 70.00%)
12 Not Duplicate (Confidence: 60.29%)
13 Not Duplicate (Confidence: 55.14%)
14 Duplicate (Confidence: 68.06%)
